# 🦀 Day 6: Attribute Macros
---

Today we explore **attribute macros** — `#[my_attr]` on functions, structs, and more.

- **What they are**: Macros that modify or replace the annotated item
- **Difference from derive**: Can change the item, not just add impls
- **Signature**: `(attr: TokenStream, item: TokenStream) -> TokenStream`
- **#[log_calls]**: Wrap functions with logging
- **#[route("GET", "/path")]**: Web framework routing
- **Parsing attribute arguments** and conditional compilation

## Attribute Macro Signature

```rust
#[proc_macro_attribute]
pub fn my_attr(attr: TokenStream, item: TokenStream) -> TokenStream {
    // attr = tokens in #[my_attr(...)]
    // item = the annotated function/struct/etc.
    item  // or modified version
}
```

Usage: `#[my_attr] fn foo() {}` or `#[my_attr(arg1, arg2)] fn foo() {}`

In [ ]:
// #[log_calls] — wraps a function to log before/after each call
// In proc-macro crate:
//
// #[proc_macro_attribute]
// pub fn log_calls(_attr: TokenStream, item: TokenStream) -> TokenStream {
//     let input = syn::parse_macro_input!(item as syn::ItemFn);
//     let fn_name = &input.sig.ident;
//     let block = &input.block;
//     let output = quote::quote! {
//         fn #fn_name() {
//             println!("Calling {:?}", stringify!(#fn_name));
//             #block
//             println!("Done {:?}", stringify!(#fn_name));
//         }
//     };
//     output.into()
// }
//
// User: #[log_calls] fn greet() { println!("Hi!"); }
// Expands to: fn greet() { println!("Calling greet"); println!("Hi!"); println!("Done greet"); }

println!("Attribute macros wrap or replace the annotated item.");

In [ ]:
// #[route("GET", "/path")] — web framework pattern
// attr = "GET", "/path" — parse with syn::parse_str or custom parser
//
// #[proc_macro_attribute]
// pub fn route(attr: TokenStream, item: TokenStream) -> TokenStream {
//     let args = syn::parse_macro_input!(attr as RouteArgs);  // custom struct
//     let input = syn::parse_macro_input!(item as syn::ItemFn);
//     // Register route at compile time or generate match arms
//     quote::quote! { ... }.into()
// }
//
// User: #[route("GET", "/users")] fn get_users() -> Json<Vec<User>> { ... }

println!("Route macros parse method and path from attr.");

In [ ]:
// Parsing attribute arguments
// Use syn::parse2(attr.into()) with a custom struct:
// struct RouteArgs { method: LitStr, path: LitStr }
// impl Parse for RouteArgs {
//     fn parse(input: ParseStream) -> Result<Self> {
//         let method = input.parse()?;
//         input.parse::<Token![,]>()?;
//         let path = input.parse()?;
//         Ok(RouteArgs { method, path })
//     }
// }

// Conditional compilation: use cfg! or #[cfg] in generated code
// quote! { #[cfg(debug_assertions)] println!("Debug: ..."); #block }

println!("Parse attr with syn::Parse; use cfg for conditional code.");

## Complete Crate Structure

```
my_attrs/
├── Cargo.toml   # proc-macro = true, syn, quote
└── src/lib.rs   # #[proc_macro_attribute] fn log_calls(...)
```

## 📝 Exercise: Design #[timed] macro

Design a `#[timed]` macro that measures function execution time.

Usage: `#[timed] fn slow_fn() { ... }` should print elapsed time when the function returns.

Hint: wrap the function body with `let start = Instant::now(); ... eprintln!("took {:?}", start.elapsed());`

In [ ]:
// YOUR CODE HERE — design #[timed]:
// - Parse ItemFn
// - Wrap block with std::time::Instant::now() and elapsed()
// - Handle different signatures (args, return type)

## 🎯 Key Takeaways

1. **Attribute macros** receive `(attr, item)` — the attribute args and the annotated item
2. They can **modify or replace** the item (unlike derive, which only adds)
3. **#[log_calls]** wraps function bodies with logging
4. **#[route("GET", "/path")]** parses method and path from `attr`
5. Parse `attr` with `syn::parse2` and a custom `Parse` impl
6. Use `#[cfg]` in generated code for conditional compilation

---
**Tomorrow:** Mini-project — Testing framework macro